# Hybrid BP + Edge-Aware GAT Decoder

This notebook trains the differentiable sum-product BP plus edge-aware GAT decoder on Google Colab. It uses the LDPC parity-check matrices already present in `GAT+BP/Codes_DB` and reports BP and hybrid BER/FER together.

Before running this notebook, push the latest `GAT+BP` code to GitHub or change `REPO_URL` below to the repository containing your code.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = 'https://github.com/gouravanirudh05/SRIP_LDPC_Decoding_using_Machine_Learning.git'
REPO_DIR = Path('/content/SRIP_LDPC_Decoding_using_Machine_Learning')
GAT_DIR = REPO_DIR / 'GAT+BP'

if not (GAT_DIR / 'train_gat_bp.py').exists():
    if REPO_DIR.exists():
        raise FileNotFoundError(
            f'{GAT_DIR / "train_gat_bp.py"} is missing. ' 
            'Remove the incomplete clone or set REPO_URL to the correct repository.'
        )
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

if not (GAT_DIR / 'train_gat_bp.py').exists():
    raise FileNotFoundError('GAT+BP/train_gat_bp.py was not found in the cloned repository.')

os.chdir(GAT_DIR)
if str(GAT_DIR) not in sys.path:
    sys.path.insert(0, str(GAT_DIR))
print('Working directory:', Path.cwd())
print('Training script:', GAT_DIR / 'train_gat_bp.py')

In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU is not enabled. In Colab select Runtime > Change runtime type > T4 GPU for practical training speed.')

## Training configuration

The available LDPC matrices include `(49,24)`, `(121,60)`, `(121,70)`, and `(121,80)`. Start with `(49,24)` to verify the pipeline, then increase the code size or training budget.

In [ ]:
import json
import time

CODE_TYPE = 'LDPC'
CODE_N = 49
CODE_K = 24
BP_ITERATIONS = 5
HIDDEN_DIM = 32
HEADS = 2
HEAD_DIM = 16

STEPS = 5000
BATCH_SIZE = 256
LEARNING_RATE = 2e-4
TRAIN_EBN0_MIN = 2.0
TRAIN_EBN0_MAX = 6.0

EVAL_EBN0 = ['2', '3', '4', '5', '6']
EVAL_EVERY = 500
EVAL_BATCHES = 50
EVAL_BATCH_SIZE = 2048

SAVE_DIR = Path('/content/Results_GAT_BP')
LOG_FILE = Path('/content/gat_bp_training.log')
print('Checkpoint directory:', SAVE_DIR)
print('Code:', f'{CODE_TYPE} (n={CODE_N}, k={CODE_K})')

In [ ]:
command = [
    sys.executable, 'train_gat_bp.py',
    '--code-type', CODE_TYPE,
    '--code-n', str(CODE_N),
    '--code-k', str(CODE_K),
    '--bp-iterations', str(BP_ITERATIONS),
    '--hidden-dim', str(HIDDEN_DIM),
    '--heads', str(HEADS),
    '--head-dim', str(HEAD_DIM),
    '--steps', str(STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--lr', str(LEARNING_RATE),
    '--train-ebn0-min', str(TRAIN_EBN0_MIN),
    '--train-ebn0-max', str(TRAIN_EBN0_MAX),
    '--eval-ebn0', *EVAL_EBN0,
    '--eval-every', str(EVAL_EVERY),
    '--eval-batches', str(EVAL_BATCHES),
    '--eval-batch-size', str(EVAL_BATCH_SIZE),
    '--device', DEVICE,
    '--save-dir', str(SAVE_DIR),
    '--log-file', str(LOG_FILE),
]
print('Running:', ' '.join(command))
started = time.perf_counter()
run = subprocess.run(
    command,
    cwd=str(GAT_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
training_output = run.stdout
print(training_output)
print('Log file:', LOG_FILE)
print(f'Wall time: {(time.perf_counter() - started) / 3600:.2f} hours')
if run.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {run.returncode}. See {LOG_FILE}.')

In [ ]:
import matplotlib.pyplot as plt

evaluation_records = []
for line in training_output.splitlines():
    try:
        record = json.loads(line)
    except json.JSONDecodeError:
        continue
    if 'evaluation' in record:
        evaluation_records.append(record)

if not evaluation_records:
    raise RuntimeError('No evaluation records were found in the training output.')

last_evaluation = evaluation_records[-1]['evaluation']
ebn0 = sorted(float(value) for value in last_evaluation)
metrics = [last_evaluation[f'{value:g}'] for value in ebn0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for key, label, style in [
    ('bp_ber', 'BP', '--o'),
    ('gat_bp_ber', 'BP + edge-aware GAT', '-o'),
]:
    axes[0].semilogy(ebn0, [row[key] for row in metrics], style, label=label)
for key, label, style in [
    ('bp_fer', 'BP', '--o'),
    ('gat_bp_fer', 'BP + edge-aware GAT', '-o'),
]:
    axes[1].semilogy(ebn0, [row[key] for row in metrics], style, label=label)

axes[0].set_title(f'BER: {CODE_TYPE}({CODE_N},{CODE_K})')
axes[1].set_title(f'FER: {CODE_TYPE}({CODE_N},{CODE_K})')
for axis, ylabel in zip(axes, ['Bit error rate', 'Frame error rate']):
    axis.set_xlabel('Eb/N0 (dB)')
    axis.set_ylabel(ylabel)
    axis.grid(True, which='both', alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.show()

print(json.dumps(last_evaluation, indent=2, sort_keys=True))

In [ ]:
checkpoint = SAVE_DIR / 'best_model.pt'
print('Checkpoint:', checkpoint)
print('Size (MB):', checkpoint.stat().st_size / (1024 ** 2))

# Uncomment this cell to download the trained checkpoint to your computer.
# from google.colab import files
# files.download(str(checkpoint))